# Clean up results
This part of the script attempts to clean up ocr mess

Though its your effort for most part 🥹

In [1]:
from pathlib import Path
debug_crops_folder="debug_crops" #edit me
path = Path(debug_crops_folder)
csv_file = "result.csv" #edit me
assert path.exists()

In [2]:
import pandas as pd
import numpy as np
import cv2
import opencv_jupyter_ui as jcv2
from objects.cropped_frame import CroppedFrame #will see some warning
import pyperclip

df = pd.read_csv(csv_file)

def clean_damage_ba(val):
    if pd.isna(val): return np.nan
    val = str(val).replace('"', '').replace(',', '').strip()
    # Fix common video OCR typos (like letters substituted for numbers)
    val = val.lower().replace('t', '1')
    if val.endswith('k'):
        return float(val[:-1]) * 1000
    return float(val)

def clean_damage_pct(val):
    if pd.isna(val): return np.nan
    val = float(val)
    if val > 100: return val/10 #miss out decimal point
    return val

def get_timestamp_of_interest(interval_dmg,dmg_numeric, type = "ba"):
    df_negative_damage = df[df[interval_dmg] < 0]
    df_negative_damage = df_negative_damage[df_negative_damage[dmg_numeric] > 0]
    if type == "pct":
        df_negative_damage = df_negative_damage[df_negative_damage[interval_dmg] !=-100]
    print(df_negative_damage.shape[0],"rows with negative damage","in",interval_dmg)
    if df_negative_damage.shape[0] == 0:
        return 0
    return df_negative_damage.index[0]



Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
C:\Users\pigut\PycharmProjects\damage_analyser\.venv\Lib\site-packages\torch\ao\nn\quantized\dynamic\modules\rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\quantized\Quantizer.cpp:116.)
  w_ih = torch.quantize_per_tensor(


## Doing some checks on which ba damage looks off
some expected issues which need datafix
- ocr reads characters wrongly
    - e.g. '8' as 'b'
    - this will throw **red text (error)** in the following cell
- missing characters from ocr
    - e.g. 183,234,224k as 1**0**3,234,224k
    - may show up as negative damage in chart
    - not necessarily detectable

<a id=ba_error_checking> BA error checking</a>

In [3]:
# checking for conversion error
row_of_interest=0
for idx, row in df.iterrows():
    try:
        clean_damage_ba(row['dmg_ba'])
    except Exception as e:
        pyperclip.copy(idx)
        row_of_interest=idx
        print(f"Failed at index {idx}, index copied on clipboard")
        print("timestamp:",df.loc[row_of_interest]["timestamp"])
        print("ba damage:",df.loc[row_of_interest]["dmg_ba"])
        break
print("All Good!")

All Good!


Found some error? [Diagnose it](#ba_diagnostics)

<a id=ba_negative_checking> BA negative checking</a>

In [4]:
df['dmg_ba_numeric'] = df['dmg_ba'].apply(clean_damage_ba)
df['dmg_ba_numeric'] = df['dmg_ba_numeric'].interpolate()
df['interval_dmg'] = df['dmg_ba_numeric'].diff().fillna(0)

row_of_interest=get_timestamp_of_interest("interval_dmg","dmg_ba_numeric")
if row_of_interest != 0:
    print("timestamp:",df.loc[row_of_interest]["timestamp"])
    print("ba damage:",df.loc[row_of_interest]["dmg_ba"])
    print("interval damage:",df.loc[row_of_interest]["interval_dmg"])
else:
    print("All Good!")

111 rows with negative damage
timestamp: 345.96
ba damage: 651,980,501k
interval damage: -80000.0


#### <a id=ba_diagnostics> BA Diagnostics </a>
*q: why is index column `index` instead of `timestamp`? all `timestamp` should be unique*

a: sometimes its the previous row causing the error e.g.
- row 1: dmg_ba = 18❌ (correct reading 10)
- row 2: dmg_ba = 10✅

In this case, row 2 was flagged because 10-18 < 0 🚩 and we should be checking both row 2 and row 1

In [5]:
# row_of_interest = 0 #edit me
print(row_of_interest)

228


In [7]:
row_of_interest+=1
print(row_of_interest)

579


In [25]:
row_of_interest-=1
print(row_of_interest)

597


In [5]:
timestamp_of_interest = df['timestamp'][row_of_interest]
assert row_of_interest != 0, "Row 0 could not be checked. Theres may be no issues or theres conversion issue with row 0"
print("timestamp:",timestamp_of_interest)
pyperclip.copy(df.loc[row_of_interest]["dmg_ba"])
print(row_of_interest-1,"dmg_ba:",df.loc[row_of_interest-1]["dmg_ba"])
print(row_of_interest,"dmg_ba:",df.loc[row_of_interest]["dmg_ba"], "Copied!")
print(row_of_interest+1,"dmg_ba:",df.loc[row_of_interest+1]["dmg_ba"])
frame = cv2.imread(path/f'{timestamp_of_interest:.2f}_dmg_ba.png')
if frame is not None and frame.size > 0:
    zoomed_frame = cv2.resize(frame, None, fx=4, fy=4, interpolation=cv2.INTER_LINEAR)
    jcv2.imshow('frame', zoomed_frame)
    cropped_frame = CroppedFrame(frame,"ba","dmg_ba")
    readings = cropped_frame.read_frame()
    print("EasyOCR Readings:",readings[0])
    print("EasyOCR Confidence:",readings[1])

timestamp: 345.96
227 dmg_ba: 651,980,581k
228 dmg_ba: 651,980,501k Copied!
229 dmg_ba: 651,980,501k


C:\Users\pigut\PycharmProjects\damage_analyser\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


EasyOCR Readings: 651,980,581k
EasyOCR Confidence: 0.999


In [27]:
df.at[row_of_interest, "dmg_ba"] = #"2,001,631,966,175k" #edit me
df.to_csv(csv_file, index=False)
df = pd.read_csv(csv_file)

back to
- [ba error checking](#ba_error_checking)
- [ba negative checking](#ba_negative_checking)

## Doing some checks on which percentage damage looks off
similar to ba damage,

some expected issues which need datafix

<a id=pct_error_checking> Percentage error checking</a>

In [6]:
# checking for conversion error
row_of_interest=0
err = False
for header_name in df.columns:
    if not header_name.endswith("_pct"):
        continue
    for idx, row in df.iterrows():
        try:
            clean_damage_pct(row[header_name])
        except Exception as e:
            err = True
            pyperclip.copy(idx)
            row_of_interest=idx
            print(f"Failed at index {idx}, index copied on clipboard")
            print("timestamp:",df.loc[row_of_interest]["timestamp"])
            print(header_name,"damage:",df.loc[row_of_interest][header_name])
            break
    if err:
        break
print("All Good!")

All Good!


<a id=pct_negative_checking> Percentage negative checking</a>


In [193]:
header_name=""
numeric_header_name = ""
interval_header_name = ""
for header_name in df.columns:
    if not header_name.endswith("_pct"):
        continue
    numeric_header_name = f'{header_name}_numeric'
    interval_header_name = f'{header_name}_interval'
    df[numeric_header_name] = df[header_name].apply(clean_damage_pct)
    df[interval_header_name] = -df[numeric_header_name].diff().fillna(0)

    row_of_interest=get_timestamp_of_interest(interval_header_name,numeric_header_name,type="pct")
    if row_of_interest != 0:
        break
if row_of_interest != 0 and header_name:
    print("timestamp:",df.loc[row_of_interest]["timestamp"])
    print(f"{numeric_header_name} damage:",df.loc[row_of_interest][numeric_header_name])
    print("interval damage:",df.loc[row_of_interest][interval_header_name])
else:
    print("All Good!")

0 rows with negative damage
0 rows with negative damage
0 rows with negative damage
4 rows with negative damage
timestamp: 450.02
dog_pct_numeric damage: 99.6
interval damage: -99.6


In [190]:
row_of_interest+=1
print(row_of_interest)

334


In [184]:
row_of_interest-=1
print(row_of_interest)

348


In [194]:
timestamp_of_interest = df['timestamp'][row_of_interest]
assert row_of_interest != 0, "Row 0 could not be checked. Theres may be no issues or theres conversion issue with row 0"
assert header_name, "'_pct' header not found"
print("timestamp:",timestamp_of_interest)
pyperclip.copy(df.loc[row_of_interest][header_name])
print(row_of_interest-1,f"{header_name}:",df.loc[row_of_interest-1][header_name])
print(row_of_interest,f"{header_name}:",df.loc[row_of_interest][header_name], "Copied!")
print(row_of_interest+1,f"{header_name}:",df.loc[row_of_interest+1][header_name])
frame = cv2.imread(path/f'{timestamp_of_interest:.2f}_{header_name}.png')
if frame is not None and frame.size > 0:
    zoomed_frame = cv2.resize(frame, None, fx=10, fy=10, interpolation=cv2.INTER_LINEAR)
    jcv2.imshow('frame', zoomed_frame)
    cropped_frame = CroppedFrame(frame,"pct",header_name)
    readings = cropped_frame.read_frame()
    print("EasyOCR Readings:",readings[0])
    print("EasyOCR Confidence:",readings[1])

timestamp: 450.02
332 dog_pct: 0.0
333 dog_pct: 99.6 Copied!
334 dog_pct: 98.7


EasyOCR Readings: 996
EasyOCR Confidence: 0.999


C:\Users\pigut\PycharmProjects\damage_analyser\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [192]:
df.at[row_of_interest, header_name] = 98.7 # edit me
df.to_csv(csv_file, index=False)
df = pd.read_csv(csv_file)

go back to:
- [Percentage error checking](#pct_error_checking)
- [Percentage negative checking](#pct_negative_checking)